In [ ]:
#load package
library(dplyr)
library(Biostrings)
library(ggplot2)
library(dplyr)
library(data.table)
library(rtracklayer)
library(dplyr)
library(ggbio)
library(reshape2)
library(ggsci)
library(viridis)
library(ggpubr)
library(ggh4x)
library(khroma)
library(qs)
MySplit = function(str,sep, n){
  return(unlist(lapply(strsplit(str,sep),"[[",n)))
}

In [ ]:
anotdf = read.csv("all_ani_total_25_08_20.csv")
tax_df = read.csv("refdata/taxon_class_08_21.csv")
anotdf = merge(anotdf,tax_df,by = "species")
snpdf = qread("/cluster/home/liuhengxin/P_CAP-seq/data/combined_snps.qs")
color_vector = read.csv(file = "refdata/color_assign_new.csv")

## Figure3b UMAP

In [ ]:
#build ANI similarity matrix using skani
query_dir="contigs"
ref_dir="refdata"

work_dir="/home/xily01/lhx/skani_analysis/skani_analysis_$(date +%Y%m%d_%H%M%S)"
mkdir -p "${work_dir}"/{query_list,ref_list,results}

find "${query_dir}" -type f -name "*.fa" -o -name "*.fasta" -o -name "*.fna" | sort > "${work_dir}/query_list/query_files.txt"
find "${ref_dir}" -type f -name "*.fa" -o -name "*.fasta" -o -name "*.fna" | sort > "${work_dir}/ref_list/ref_files.txt"

# skani calling
skani dist --ql ${work_dir}/query_list/query_files.txt --rl ${work_dir}/ref_list/ref_files.txt -t ${SLURM_CPUS_PER_TASK} -s 0 --min-af 0 --no-learned-ani --no-marker-index -c 30 -m 200 -o ${work_dir}/results/all_pairs_thred10.ani

In [ ]:
#build ANI similarity matrix
anidf = qread("all_pairs_thred10.ani.withan.qs")
length(unique(anotdf$name))
length(unique(anidf$name))
anidf.max = anidf %>% group_by(name) %>% filter(ANI == max(ANI))
summary(anidf.max$ANI)
anidf.max = anidf.max[anidf.max$ANI > 95,]
anidf_fil = anidf[anidf$name %in% anidf.max$name,]
anidf_fil = anidf_fil[!is.na(anidf_fil$name),]

In [ ]:
library(reshape2)
animx = dcast(anidf_fil[anidf_fil$Align_fraction_query > 15,], new_name~Ref_name,value.var = "ANI",fun.aggregate = sum)

rownames(animx) = animx$new_name;animx = animx[,-1]
scaled_data = scale(animx)
library(uwot)
set.seed(42)
umap_result = umap(
  scaled_data, 
  n_neighbors = 20, 
  min_dist = 0.25, 
  n_components = 2,
  metric = "cosine"
)

umap_df = data.frame(UMAP1 = umap_result[,1], UMAP2 = umap_result[,2], cell_id = rownames(umap_result))
#umap_df_an$class = MySplit(umap_df_an$species," ",1)

In [ ]:
umap_df_an = merge(umap_df,anotdf[,c("name","species","sample","ANI")],by.x = "cell_id",by.y = "name")
umap_df_an = merge(umap_df_an,tax_df,by = "species",all = T)

cat("class number:", length(unique(na.omit(umap_df_an$order))))
umap_df.st = umap_df_an %>% group_by(order,species) %>% summarise(celln = length(unique(cell_id)))
umap_df.st$prop = umap_df.st$celln/sum(umap_df.st$celln)
umap_df.st = umap_df.st[!umap_df.st$species %in% c("unclassfied","unclassfied_bacteria"),]
umap_df.st = umap_df.st[umap_df.st$celln > 10,]
umap_df.st[is.na(umap_df.st$order),]
umap_df_hub = umap_df_an[umap_df_an$species %in% umap_df.st$species,]

In [ ]:
library(RColorBrewer)
umap_df_hub = umap_df_hub[order(umap_df_hub$order),]
base_colors = colorRampPalette(brewer.pal(12, "Set1"))(14)
names(base_colors) = unique(umap_df_hub$order)  

color_list = list()
for (cls in unique(umap_df_hub$class)) {

  orders_in_class = unique(umap_df_hub$order[umap_df_hub$class == cls])
  n_orders = length(orders_in_class)

  if (n_orders > 1) {
    color_grad = colorRampPalette(c(base_colors[cls], "white"))(n_orders + 1)[1:n_orders]
  } else {
    color_grad = base_colors[cls]
  }
  names(color_grad) = orders_in_class
  color_list = c(color_list, color_grad)
}

color_vector = unlist(color_list)
color_vector
length(color_vector)

In [ ]:
umap_df_hub$order = factor(umap_df_hub$order, levels = names(color_vector)[names(color_vector) %in% unique(umap_df_hub$order)])
umap_df.st = umap_df_an %>% group_by(order,species) %>% summarise(celln = length(unique(cell_id)))
umap_df.st$prop = umap_df.st$celln/sum(umap_df.st$celln)
umap_df.st = umap_df.st[umap_df.st$celln > 30,]
umap_df_hub = umap_df_an[umap_df_an$order %in% umap_df.st$order,]

umap_df_hub = umap_df_hub[umap_df_hub$ANI > 95,]
nrow(umap_df_hub)
p3.b = ggplot(umap_df_hub, aes(x = UMAP1, y = UMAP2,color = order)) +
  geom_point(size = 0.1, alpha = 0.8) +
 geom_point(data = umap_df_hub[umap_df_hub$species == "Clostridioides difficile",],
            aes(x = UMAP1, y = UMAP2),color = "red",shape = 5, size = 2, alpha = 0.8) + 
  theme_minimal() +
  scale_color_manual(values = color_vector) +
  labs(title = "UMAP Projection of Cell-Microbe ANI Similarity") + 
  #facet_wrap(~sample) +
  theme(legend.position = "bottom") +
  guides(color = guide_legend(override.aes = list(size=4),
                                  nrow = 3),size = "none")
p3.b

In [ ]:
p3s.e = ggplot(umap_df_hub[umap_df_hub$ANI>95,], aes(x = UMAP1, y = UMAP2,color = group)) +
  geom_point(size = 0.1, alpha = 0.8) +
    facet_wrap(~sample) +
  theme_pubr() +
  scale_color_brewer(palette = "Set2") +
  labs(title = "UMAP Projection of Different sample") + 
  theme(legend.position = "bottom") +
  guides(color = guide_legend(override.aes = list(size=4,nrow = 1)),size = "none")
p3s.e

p3s.f = ggplot(umap_df_hub[umap_df_hub$ANI>95,], aes(x = UMAP1, y = UMAP2,color = condition)) +
  geom_point(size = 0.1, alpha = 0.8) +
  theme_pubr() +
  scale_color_brewer(palette = "Set2") +
  labs(title = "UMAP Projection of Different sample") + 
  theme(legend.position = "bottom") +
  guides(color = guide_legend(override.aes = list(size=4,nrow = 1)),size = "none")
p3s.f

## Figure3c Species tree

In [ ]:
#build ANI matrix between all SAGs and refs
skani triangle -l ref_files.txt -t 32 --full-matrix --min-af 0  --robust -s 0 -o ani_matrix_skani_raw_new.txt

In [ ]:
lines = readLines("ani_matrix_skani_raw_new.txt.af")
matrix_list = lapply(lines, function(line) {
  elements = scan(text = line, what = character(), quiet = TRUE)

  path = elements[1]
  values = as.numeric(elements[-1])
  list(path = path, values = values)
})

n = length(matrix_list) 
full_matrix_af = matrix(NA, nrow = n, ncol = n)
rownames(full_matrix_af) = sapply(matrix_list, `[[`, "path")

for (i in 1:n) {
  vals = matrix_list[[i]]$values
  if (length(vals) > 0) {
    full_matrix_af[i, 1:length(vals)] = vals
  }
}
library(stringr)
rownames(full_matrix_af) = str_extract(rownames(full_matrix_af), "GC[AF]_[0-9]+\\.[0-9]+")
full_matrix_af = full_matrix_af[-1,]
dim(full_matrix_af)
colnames(full_matrix_af) = c(rownames(full_matrix_af),"")

### SNP analysis

In [ ]:
#!/bin/bash

INPUT="/file_ref_match_ls.tsv"
OUTDIR="contig_snp_total"
MERGE_DIR="contig_snp_merged_results"

mkdir -p $OUTDIR
mkdir -p $MERGE_DIR

run_snippy() {
    query="$1"
    ref="$2"
    
    sample_name=$(basename "$query" .fasta)
    safe_sample_name=$(echo "$sample_name" | sed 's/\./_/g;s/[^a-zA-Z0-9_]/-/g')
    
    sample_dir="$OUTDIR/${safe_sample_name}"
    #mkdir -p "$sample_dir"
    
    snippy --cpus 2 \
           --outdir "$sample_dir" \
           --ref "$ref" \
           --ctgs "$query" \
           --quiet
           
}


tail -n +2 "$INPUT" | parallel --colsep '\t' -j $(nproc) run_snippy {1} {2}

find "$OUTDIR" -name "snps.tab" > $MERGE_DIR/snp_file_list.txt

printf "Sample\tRef_Assembly\tTotal_SNPs\n" > $MERGE_DIR/snp_counts.tsv
while read tabfile; do
    sample_dir=$(dirname "$tabfile")
    sample_name=$(basename "$sample_dir")
    ref_assembly=$(grep 'Reference:' "$tabfile" | awk '{print $2}')
    snp_fa="${sample_dir}/snps.fa"
    total_snps=$([ -f "$snp_fa" ] && grep -c '^>' "$snp_fa" || echo "0")
    printf "%s\t%s\t%d\n" "$sample_name" "$ref_assembly" "$total_snps" >> $MERGE_DIR/snp_counts.tsv
done < $MERGE_DIR/snp_file_list.txt


In [ ]:
#build snps of all SAGs
snpdf = read.delim("/cluster/home/liuhengxin/P_CAP-seq/data/combined_snps.tab")
head(snpdf)

In [ ]:
library(reshape2)
gcaan = unique(anotdf[,c("GCA","species")])
nrow(gcaan)
length(unique(gcaan$species))
length(unique(gcaan[gcaan$GCA %in% rownames(full_matrix),]$species))
anotdf.st = anotdf %>% group_by(order,species) %>% summarise(celln = length(unique(name)))
anotdf.st$prop = anotdf.st$celln/sum(anotdf.st$celln)
anotdf.st = anotdf.st[!anotdf.st$species %in% c("unclassfied","unclassfied_bacteria"),]
anotdf.st = anotdf.st[anotdf.st$celln > 10,]
anotdf.st = anotdf.st[order(-anotdf.st$celln),]
head(anotdf.st)
length(unique(anotdf.st$species))
hubsp = anotdf.st$species[1:100]
snpdf.sp = snpdf.st.an %>% group_by(species) %>% 
           summarise(mindeln = mean(indeln))

snpdf.st = snpdf[snpdf$TYPE == "snp",] %>% group_by(name) %>% summarise(indeln = length(indel))
snpdf.st$name = gsub("-", "_", snpdf.st$name)
snpdf.st.an = merge(snpdf.st,anotdf,by = "name")
nrow(snpdf.st.an)

#filter
colnames(snpdf)[15] = "name"
snpdf$name = gsub("-", "_", snpdf$name)
snpdf = merge(snpdf[,-ncol(snpdf)],anotdf[,c("name","sample","species")],by = "name")
qsave(snpdf,file = "combined_snps.qs")

### Plot tree

In [ ]:
library(reshape2)
gcaan = unique(anotdf[,c("GCA","species")])
nrow(gcaan)
length(unique(gcaan$species))
length(unique(gcaan[gcaan$GCA %in% rownames(full_matrix),]$species))
anotdf.st = anotdf %>% group_by(order,species) %>% summarise(celln = length(unique(name)))
anotdf.st$prop = anotdf.st$celln/sum(anotdf.st$celln)
anotdf.st = anotdf.st[!anotdf.st$species %in% c("unclassfied","unclassfied_bacteria"),]
anotdf.st = anotdf.st[anotdf.st$celln > 10,]
anotdf.st = anotdf.st[order(-anotdf.st$celln),]
head(anotdf.st)
length(unique(anotdf.st$species))
hubsp = anotdf.st$species[1:100]
snpdf.sp = snpdf.st.an %>% group_by(species) %>% 
           summarise(mindeln = mean(indeln))


full_df = melt(full_matrix)
full_df = merge(full_df,gcaan,by.x = "Var1",by.y = "GCA")
full_df = merge(full_df,gcaan,by.x = "Var2",by.y = "GCA")
colnames(full_df)[4:5] = c("species1","species2")
full_df = full_df[full_df$species1 %in% hubsp & full_df$species2 %in% hubsp,]
full_matrix2 = dcast(full_df,species1~species2,value.var = "value",fun.aggregate = max, fill = 0)
rownames(full_matrix2) = full_matrix2$species1; full_matrix2 = full_matrix2[,-1]
dim(full_matrix2)

In [ ]:
bar_data = data.frame(
    ID = tree$label,  # 与树上节点ID匹配
    Value = anotdf.st$celln[match(tree$label, anotdf.st$species)],
    Group = tax_df$order[match(tree$label, tax_df$species)]  # 分组列（可选）
)
bar_data[order(-bar_data$Value),]
dd = dist(full_matrix2, method = "cosine")
tree = hclust(dd, method = "complete")

species_data = data.frame(
  label = tree$label,

  order = tax_df$order[match(tree$label, tax_df$species)]
)


bar_data = data.frame(
    ID = tree$label,
    Value = anotdf.st$prop[match(tree$label, anotdf.st$species)],
    Group = tax_df$order[match(tree$label, tax_df$species)] 
)

bar_data2 = data.frame(
    ID = tree$label, 
    Value = snpdf.sp$mindeln[match(tree$label, snpdf.sp$species)],
    Group = tax_df$order[match(tree$label, tax_df$species)]  
)

back_data = data.frame(
    ID = tree$label, 
    Value = 1,
    Group = tax_df$order[match(tree$label, tax_df$species)] 
)

label_data = species_data
label_data$ID = label_data$label
species_data$order = factor(species_data$order, levels = names(color_vector)[names(color_vector) %in% unique(species_data$order)])
back_data$Group = factor(back_data$Group, levels = names(color_vector)[names(color_vector) %in% unique(back_data$Group)])
bar_data$Group = factor(bar_data$Group, levels = names(color_vector)[names(color_vector) %in% unique(bar_data$Group)])

p3c1 = ggtree(tree, layout = "circular", aes(color=order), size = 0.5) %<+% species_data +
  geom_tippoint(aes(color = order), size = 0.2) +
  geom_fruit(
      data = back_data,           
      geom = geom_bar,           
      mapping = aes(
          y = ID,                
          x = Value,             
          fill = Group           
      ),
      stat = "identity",         
      orientation = "y",         
      pwidth = 1.5, 
      alpha = 0.5,
      offset = -0.05,
  ) +
  geom_tiplab(size = 5, offset = 0.08, show.legend = FALSE, color = "black") +
  scale_color_manual(values = color_vector) +
  scale_fill_manual(values = color_vector) +
  geom_fruit(
      data = bar_data2,           
      geom = geom_point,           
      mapping = aes(
          y = ID,                
          size = Value,
          color = Group,
          alpha = Value
      ),
      pwidth = 0.8,              
      offset = 0.08
  ) +
  theme(legend.position = "bottom") +
  geom_fruit(
      data = bar_data,           
      geom = geom_bar,           
      mapping = aes(
          y = ID,                
          x = Value,             
          fill = Group
      ),
      stat = "identity",         
      orientation = "y",         
      pwidth = 0.8,              
      offset = 0.08,

      axis.params = list(
        axis = "x",text.size = 6,nbreak     = 3,
        text.color = "black"
      ),
      grid.params=list()
  )
p3c1

anotdf.st = anotdf %>% group_by(order) %>% summarise(celln = length(unique(name)))
anotdf.st$prop = anotdf.st$celln/sum(anotdf.st$celln)
unique(anotdf.st$order)
anotdf.st = anotdf.st[order(anotdf.st$prop),]
anotdf.st$order = factor(anotdf.st$order, levels = anotdf.st$order)
p3c2 = ggplot(anotdf.st[anotdf.st$order %in% names(color_vector), ],aes(x = prop+0.1, y = order, fill = order)) + geom_bar(stat = "identity") + 
    scale_fill_manual(values = color_vector) + 
    scale_x_reverse(breaks = c(0,0.4),position = "top",) + 
    scale_y_discrete(position = "right") +
    theme_pubr() + theme(legend.position = "none") + xlab("") + ylab("")

p3c2

## Figure3e-g

In [ ]:
#highlight
umap_df_hub.high = umap_df_hub[umap_df_hub$species %in% c("Bacteroides fragilis",
                                                     "Bacteroides uniformis",
                                                     "Parabacteroides distasonis"),]
umap_df_hub.back = umap_df_hub[!umap_df_hub$species %in% c("Bacteroides fragilis",
                                                     "Bacteroides uniformis",
                                                     "Parabacteroides distasonis"),]
p3d = ggplot() +
  geom_point(data = umap_df_hub.back,
              aes(x = UMAP1, y = UMAP2),color = "grey",
             size = 0.1, alpha = 0.8) +
 geom_point(data = umap_df_hub.high,
              aes(x = UMAP1, y = UMAP2,color = species),
             size = 0.1, alpha = 0.8) +
    scale_color_brewer(palette = "Set1") +
 # geom_point(data = umap_df_hub[umap_df_hub$species == "Clostridioides difficile",],
 #            aes(x = UMAP1, y = UMAP2),color = "red",size = 1, alpha = 0.8) + 
  theme_pubr() +
  #scale_color_manual(values = color_vector) +
  #facet_wrap(~sample) +
  #theme(legend.position = "none") +
  guides(color = guide_legend(override.aes = list(size=4),nrow = 3),size = "none")
p3d

In [ ]:
library(ape)
library(ggtree)
library(tidyverse)
library(treeio)
library(proxy)
library(amap)

unique_names = unique(umap_df_hub[umap_df_hub$species %in% 
                                   c("Bacteroides fragilis","Bacteroides uniformis","Parabacteroides distasonis") &
                                   umap_df_hub$ANI > 99,]$cell_id)
length(unique_names)
hubcid1 = "CGGAGCT_TAAGAGA_GGAACTT"
hubcid2 = "TATCTCA_TAGGCAT_AAATGAG"

unique_names = unique_names[sample(1:length(unique_names),size = 298)]
unique_names = c(unique_names,hubcid1,hubcid2)
length(unique_names)
anidf_hub = anidf[anidf$name %in% unique_names & anidf$Align_fraction_query > 15,]
animx_hub = dcast(anidf_hub, name~Ref_name,value.var = "ANI",fun.aggregate = sum)
rownames(animx_hub) = animx_hub$name;animx_hub = animx_hub[,-1]

dd = dist(animx_hub, method = "cosine")
tree = hclust(dd)

species_data = data.frame(
  label = tree$label, 
  species = anotdf$species[match(tree$label, anotdf$name)]

)
species_data$highlight = ""
species_data[species_data$label == hubcid1,]$highlight = hubcid1
species_data[species_data$label == hubcid2,]$highlight = hubcid2

p3e = ggtree(tree, aes(color=species), size = 0.1) %<+% species_data +
  geom_tippoint(aes(color = species), size = 0.2,align=TRUE) +
  geom_tiplab(aes(label = highlight), size = 2.5, offset = 0.02, show.legend = FALSE) +
  scale_color_brewer(palette = "Set1", name = "Species", na.value = "grey50") +
  theme(legend.position = "bottom")
p3e
ggexport(p3d,filename = "../downstream_analysis/figures/ANI_treeplot_single_cell_25_09_16.pdf",width = 3,height = 3)

## Figure 3S a-d

In [ ]:
#ANI stat
p3sa = ggplot(anidf,aes(ANI,color = sample)) + geom_density() + xlab("ANI distribution") + ylab("Cell counts") + theme_pubr() +
scale_color_discreterainbow()
p3sa

In [ ]:
library(tidyplots)
p3sb = anidf.st %>%
  tidyplot(x = sample, y = celln, color = condition) |> 
  add_barstack_absolute() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5,hjust = 1)) + 
  ylab("Cell numbers")
p3sb

p3sc = anidf.st %>%
  tidyplot(x = sample, y = specien, color = condition) |> 
  add_barstack_absolute() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5,hjust = 1)) + 
  ylab("Species numbers")
p3sc


In [ ]:
#Clostridioides difficile change
anidf.st = anotdf %>% group_by(sample) %>% mutate(cellnumt = length(unique(name))) %>% 
                group_by(sample,group,condition,species) %>% summarise(cellnum = length(unique(name)),
                                                       cellnumt = cellnumt[1],
                                                       cellprop = cellnum/cellnumt)
anidf.st.hub = anidf.st[anidf.st$species == "Clostridioides difficile",]
anidf.st.hub$condition = factor(anidf.st.hub$condition,levels =unique(anidf.st.hub$condition))

p3sd = ggplot(data = anidf.st.hub,aes(x = condition,y = cellnum,fill = group))+ 
    geom_point(position = position_dodge(0.5),shape = 21,size = 3) + 
    geom_bar(position = position_dodge(0.5),stat = "identity",width = 0.05) +
    geom_text(position = position_dodge(0.5),aes(label=cellnum),vjust = -1) + 
    scale_fill_npg() +
    scale_color_npg() +
    theme_pubr() + theme(axis.text.x = element_text(angle = 90,hjust = 0.5,vjust = 0.5)) + 
xlab("Treatment condition") + ylab("Number of cell")
p3sd